In [7]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

import holidays
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, make_scorer

# Building "complete" data frame

Covariates to include:
see Modelscore_log EXCEL

Data_frames needed as base:
(everything in the raw_data folder)

In [4]:
wd = !pwd
print(f'Your current working directory is {wd[0]}')

Your current working directory is /Users/laurenzpfleiderer/code/xucenying/grid-intelligence/notebooks/laurenz


In [5]:
path = "../../raw_data/"
file_price = "combined_energy_price_clean.csv"
file_generation = "encoded_netgen.csv"
file_weather = "weather_avg.csv"
file_fuel = "oil_gas_prices.csv"
file_consumption = "load_data.csv"

In [8]:
data_price = pd.read_csv(f'{path}{file_price}', sep='\t')
data_generation = pd.read_csv(f'{path}{file_generation}')
data_weather = pd.read_csv(f'{path}{file_weather}')
data_fuel = pd.read_csv(f'{path}{file_fuel}')
data_consumption = pd.read_csv(f'{path}{file_consumption}')

/var/folders/sj/pc75w_ds6w72_p6qqgqy9m980000gn/T/ipykernel_90275/981727959.py:1: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  data_price = pd.read_csv(f'{path}{file_price}', sep='\t')


### Cleaning functions

In [25]:
def prepare_price_data(df):
    """
    Filters sequence 2 (15-min intervals) and returns a clean dataframe
    with columns: ['price', 'date'], date being Datetime type.
    """
    # choosing sequence 2 for 15 minute intervals
    df_2 = df[df["Sequence"] == 2].copy()

    df_2 = df_2.drop(columns="Sequence")

    df_2["DateTime(UTC)"] = pd.to_datetime(
        df_2["DateTime(UTC)"],
        format="%Y-%m-%d %H:%M:%S"
    )

    df_2 = df_2.rename(columns={"Price[Currency/MWh]": "price"})

    df_2 = df_2[["DateTime(UTC)","price"]]

    return df_2

In [ ]:
data_price_cleaned = prepare_price_data(data_price)

In [27]:
def prepare_generation_data(df):
    df_2 = df.copy()

    df_2["DateTime(UTC)"] = pd.to_datetime(
        df_2["DateTime(UTC)"],
        format="%Y-%m-%d %H:%M:%S"
    )

    df_2 = df_2[['DateTime(UTC)', 'other', 'other_renewable', 'stable', 'variable']]

    return df_2

In [28]:
data_generation_cleaned = prepare_generation_data(data_generation)

In [57]:
def prepare_consumption_data(df):
    """
    Converts Berlin timezone timestamps (with DST) to UTC,
    fills missing timestamps caused by DST shifts using forward fill.

    Parameters:
        df (pd.DataFrame): must contain columns ['Unnamed: 0', 'load']

    Returns:
        pd.DataFrame with ['DateTime(UTC)', 'load']
    """

    data = df.copy()

    # rename
    data = data.rename(columns={"Unnamed: 0": "DateTime"})

    # parse datetime safely
    data["DateTime"] = pd.to_datetime(data["DateTime"], errors="coerce")

    # FORCE timezone handling (no conditional logic needed)
    data["DateTime"] = data["DateTime"].dt.tz_localize(
        "Europe/Berlin",
        ambiguous="infer",
        nonexistent="shift_forward"
    )

    # convert to UTC
    data["DateTime(UTC)"] = data["DateTime"].dt.tz_convert("UTC")

    # remove timezone info for consistency with other datasets
    data["DateTime(UTC)"] = data["DateTime(UTC)"].dt.tz_localize(None)

    # keep only relevant columns
    data = data[["DateTime(UTC)", "load"]]

    data = data.sort_values("DateTime(UTC)").set_index("DateTime(UTC)")

    # build complete 15-min grid
    full_range = pd.date_range(
        start=data.index.min(),
        end=data.index.max(),
        freq="15min"
    )

    data = data.reindex(full_range)

    # impute missing (DST gaps etc.)
    data["load"] = data["load"].ffill()

    return data.reset_index().rename(columns={"index": "DateTime(UTC)"})

In [58]:
data_consumption_clean = prepare_consumption_data(data_consumption)

/var/folders/sj/pc75w_ds6w72_p6qqgqy9m980000gn/T/ipykernel_90275/3586735361.py:19: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  data["DateTime"] = pd.to_datetime(data["DateTime"], errors="coerce")


AttributeError: Can only use .dt accessor with datetimelike values

### Merging functions

In [29]:
def merge_time_series_with_weather(
    main_df,
    weather_df,
    datetime_col="DateTime(UTC)",
    strategy="ffill"  # "ffill" or "interpolate"
    ):
        # -------------------------
    # 1. Convert datetime
    # -------------------------
    main_df = main_df.copy()
    weather_df = weather_df.copy()

    main_df[datetime_col] = pd.to_datetime(main_df[datetime_col])
    weather_df[datetime_col] = pd.to_datetime(weather_df[datetime_col])

    # -------------------------
    # 2. Set index
    # -------------------------
    main_df = main_df.set_index(datetime_col).sort_index()
    weather_df = weather_df.set_index(datetime_col).sort_index()

    # -------------------------
    # 3. Reindex weather to match main timeline
    # -------------------------
    weather_aligned = weather_df.reindex(main_df.index)

    # -------------------------
    # 4. Imputation strategy
    # -------------------------
    if strategy == "ffill":
        weather_aligned = weather_aligned.ffill()

    elif strategy == "interpolate":
        weather_aligned = weather_aligned.interpolate(method="time")

    else:
        raise ValueError("strategy must be either 'ffill' or 'interpolate'")

    # -------------------------
    # 5. Merge
    # -------------------------
    merged = main_df.join(weather_aligned)

    return merged.reset_index()

In [30]:
data_merged_1 = merge_time_series_with_weather(
    data_price_cleaned,
    data_weather,
    strategy="interpolate"
)

In [49]:
def build_merged_dataset(data_merged_1, data_generation_cleaned, data_consumption,
    start_date=None, end_date=None):
    """
    Merges price/weather data with generation and consumption data. (all 15 minutes data)

    - Sets DateTime(UTC) as index
    - Optionally checks for NaNs between start_date and end_date
    """

    # --- Copy to avoid side effects
    df_main = data_merged_1.copy()
    df_gen = data_generation_cleaned.copy()
    df_cons = data_consumption.copy()

    # --- Ensure datetime format
    df_main["DateTime(UTC)"] = pd.to_datetime(df_main["DateTime(UTC)"])
    df_gen["DateTime(UTC)"] = pd.to_datetime(df_gen["DateTime(UTC)"])
    df_cons["DateTime(UTC)"] = pd.to_datetime(df_cons["DateTime(UTC)"])

    # --- Set index
    df_main = df_main.set_index("DateTime(UTC)")
    df_gen = df_gen.set_index("DateTime(UTC)")
    df_cons = df_cons.set_index("DateTime(UTC)")

    # --- Merge (left join keeps main timeline)
    merged_df = df_main.join(df_gen, how="left")
    merged_df = merged_df.join(df_cons, how="left")

    # --- Optional: check NaNs in date range
    if start_date is not None:
        start_date = pd.to_datetime(start_date)

        if end_date is not None:
            end_date = pd.to_datetime(end_date)
            df_check = merged_df.loc[start_date:end_date]
        else:
            df_check = merged_df.loc[start_date:]

        na_counts = df_check.isna().sum()
        total_na = na_counts.sum()

        print("---- NA CHECK ----")
        print(f"Rows checked: {len(df_check)}")
        print(f"Total missing values: {total_na}")
        print("Missing per column:")
        print(na_counts[na_counts > 0])

    return merged_df

In [50]:
data_merged_2 = build_merged_dataset(data_merged_1, data_generation_cleaned, data_consumption,
    start_date="2025-02-01 00:00:00", end_date="2026-02-01 00:00:00")

KeyError: 'DateTime(UTC)'

In [53]:
data_consumption_clean.head()

,DateTime(UTC),load
0,2024-01-21 00:00:00+00:00,52262.78
1,2024-01-21 00:15:00+00:00,51294.32
2,2024-01-21 00:30:00+00:00,50837.16
3,2024-01-21 00:45:00+00:00,50536.96
4,2024-01-21 01:00:00+00:00,50052.80


In [54]:
data_consumption.head()

,Unnamed: 0,load
0,2024-01-21 00:00:00+01:00,52262.78
1,2024-01-21 00:15:00+01:00,51294.32
2,2024-01-21 00:30:00+01:00,50837.16
3,2024-01-21 00:45:00+01:00,50536.96
4,2024-01-21 01:00:00+01:00,50052.80
